# Week1 복습과제_신아영
## ResNet 코드 실습
- 교재: **딥러닝 파이토치 교과서**
- 범위: **6.1.5 ResNet (p.312~340)**
- 주제: Residual Block, BasicBlock, Bottleneck, ResNet 구성 및 학습/평가

> 교재의 흐름을 따라 작성했고, 주요 코드에는 역할을 이해할 수 있도록 주석을 추가했습니다.  
> 교재의 dogs-vs-cats 데이터셋 경로가 없는 환경에서도 구조 확인 셀은 실행되도록 작성했습니다.


## 1. ResNet 핵심 개념

깊은 신경망은 층이 깊어진다고 항상 성능이 좋아지는 것은 아닙니다.  
ResNet은 **잔차 블록(residual block)**과 **숏컷 연결(shortcut / skip connection)**을 도입하여

\[
H(x) = F(x) + x
\]

형태로 학습합니다.

- `F(x)`: 합성곱 층들이 학습하는 잔차(residual)
- `x`: 입력을 그대로 전달하는 identity shortcut
- 입력과 출력의 크기/채널이 다르면 `1×1 convolution`을 사용해 차원을 맞춤

교재 p.312~317의 개념 설명을 바탕으로 정리했습니다.


## 2. 필요한 라이브러리 호출

In [1]:
!pip install -q kagglehub

In [2]:
import kagglehub

path = kagglehub.dataset_download(
    "shaunthesheep/microsoft-catsvsdogs-dataset"
)

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'microsoft-catsvsdogs-dataset' dataset.
Path to dataset files: /kaggle/input/microsoft-catsvsdogs-dataset


In [3]:
import os

print(os.listdir(path))

['PetImages', 'readme[1].txt', 'MSR-LA - 3467.docx']


In [4]:
print(os.listdir(os.path.join(path, "PetImages")))

['Dog', 'Cat']


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data

import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models

import matplotlib.pyplot as plt
import numpy as np

import copy
from collections import namedtuple
import os
import random
import time

from torch.utils.data import DataLoader, Dataset
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", device)

device: cuda


### 실행 결과 해석
GPU가 사용 가능하면 `cuda`, 그렇지 않으면 `cpu`가 출력됩니다.  
이후 모델과 입력 데이터를 같은 device로 이동시켜야 합니다.


## 3. namedtuple 사용 확인

In [6]:
Student = namedtuple('Student', ['name', 'age', 'DOB'])
S = Student('홍길동', '19', '187')

print("The Student age using index is :", S[1])
print("The Student name using keyname is :", S.name)

The Student age using index is : 19
The Student name using keyname is : 홍길동


### 실행 결과
- 인덱스(`S[1]`)로도 접근 가능
- 필드 이름(`S.name`)으로도 접근 가능

ResNet 설정값도 뒤에서 `namedtuple`로 묶어 사용합니다.


## 4. 이미지 데이터 전처리

In [7]:
class ImageTransform():
    def __init__(self, resize, mean, std):
        self.data_transform = {
            'train': transforms.Compose([
                transforms.RandomResizedCrop(resize, scale=(0.5, 1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean, std)
            ]),
            'val': transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(resize),
                transforms.ToTensor(),
                transforms.Normalize(mean, std)
            ])
        }

    def __call__(self, img, phase):
        return self.data_transform[phase](img)

size = 224
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)
batch_size = 32

print("resize:", size)
print("mean:", mean)
print("std:", std)
print("batch_size:", batch_size)

resize: 224
mean: (0.485, 0.456, 0.406)
std: (0.229, 0.224, 0.225)
batch_size: 32


### 전처리 의미
- 학습 데이터: `RandomResizedCrop`, `RandomHorizontalFlip`으로 데이터 증강
- 검증 데이터: `Resize` 후 `CenterCrop`
- ImageNet에서 많이 사용하는 평균/표준편차로 정규화


## 5. dogs-vs-cats 데이터 경로 및 Dataset 정의

In [8]:
cat_directory = os.path.join(path, "PetImages", "Cat")
dog_directory = os.path.join(path, "PetImages", "Dog")

cat_images_filepaths = [
    os.path.join(cat_directory, f)
    for f in os.listdir(cat_directory)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

dog_images_filepaths = [
    os.path.join(dog_directory, f)
    for f in os.listdir(dog_directory)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

images_filepaths = cat_images_filepaths + dog_images_filepaths

# 먼저 섞고 과제 실습에 필요한 만큼만 사용
random.seed(42)
random.shuffle(images_filepaths)

candidate_filepaths = images_filepaths[:600]

# 선택한 파일만 손상 여부 확인
correct_images_filepaths = []

for img_path in candidate_filepaths:
    try:
        with Image.open(img_path) as img:
            img.verify()
        correct_images_filepaths.append(img_path)
    except:
        pass

DATA_AVAILABLE = len(correct_images_filepaths) > 0

print("전체 후보 이미지 수:", len(images_filepaths))
print("검사한 이미지 수:", len(candidate_filepaths))
print("사용 가능한 이미지 수:", len(correct_images_filepaths))
print("DATA_AVAILABLE:", DATA_AVAILABLE)

전체 후보 이미지 수: 25000
검사한 이미지 수: 600
사용 가능한 이미지 수: 600
DATA_AVAILABLE: True


In [9]:
train_images_filepaths = correct_images_filepaths[:400]
val_images_filepaths = correct_images_filepaths[400:500]
test_images_filepaths = correct_images_filepaths[500:510]

In [10]:
class DogsvsCatsDataset(Dataset):
    def __init__(self, file_list, transform=None, phase='train'):
        self.file_list = file_list
        self.transform = transform
        self.phase = phase

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        img = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img, self.phase)

        if "Dog" in img_path:
          label = 1
        elif "Cat" in img_path:
          label = 0

        return img, label

print("DogsvsCatsDataset 정의 완료")

DogsvsCatsDataset 정의 완료


### Dataset 주석
- `__len__`: 데이터셋 크기 반환
- `__getitem__`: 특정 인덱스의 이미지와 label 반환
- cat = `0`, dog = `1`


In [11]:
if DATA_AVAILABLE:
    random.seed(42)
    random.shuffle(correct_images_filepaths)

    train_images_filepaths = correct_images_filepaths[:400]
    val_images_filepaths = correct_images_filepaths[400:500]
    test_images_filepaths = correct_images_filepaths[500:510]

    train_dataset = DogsvsCatsDataset(
        train_images_filepaths,
        transform=ImageTransform(size, mean, std),
        phase='train'
    )
    val_dataset = DogsvsCatsDataset(
        val_images_filepaths,
        transform=ImageTransform(size, mean, std),
        phase='val'
    )

    train_iterator = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True
    )
    valid_iterator = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False
    )

    batch_iterator = iter(train_iterator)
    inputs, label = next(batch_iterator)
    print(inputs.size())
    print(label)
else:
    print("데이터셋이 없어 DataLoader 실행은 생략합니다.")
    print("교재의 출력 예: torch.Size([32, 3, 224, 224])")

torch.Size([32, 3, 224, 224])
tensor([0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0,
        0, 1, 1, 1, 1, 1, 1, 0])


## 6. BasicBlock 정의 — ResNet18 / ResNet34

In [12]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=False):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels, out_channels,
            kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        # 입력 x와 F(x)의 크기/채널이 다른 경우 shortcut의 차원을 맞춤
        if downsample:
            conv = nn.Conv2d(
                in_channels, out_channels,
                kernel_size=1, stride=stride, bias=False
            )
            bn = nn.BatchNorm2d(out_channels)
            self.downsample = nn.Sequential(conv, bn)
        else:
            self.downsample = None

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(identity)

        # residual connection: F(x) + x
        out += identity
        out = self.relu(out)

        return out

print(BasicBlock(64, 64))

BasicBlock(
  (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
)


In [13]:
# BasicBlock 동작 확인
x = torch.randn(2, 64, 56, 56)

block_same = BasicBlock(64, 64)
y_same = block_same(x)

block_down = BasicBlock(64, 128, stride=2, downsample=True)
y_down = block_down(x)

print("입력:", x.shape)
print("identity shortcut:", y_same.shape)
print("downsample shortcut:", y_down.shape)

입력: torch.Size([2, 64, 56, 56])
identity shortcut: torch.Size([2, 64, 56, 56])
downsample shortcut: torch.Size([2, 128, 28, 28])


### 실행 결과 해석
- 입력/출력 차원이 같으면 identity shortcut을 그대로 더할 수 있습니다.
- 공간 크기나 채널 수가 달라지면 `1×1 convolution + BatchNorm`으로 크기를 맞춥니다.


## 7. Bottleneck 정의 — ResNet50 / 101 / 152

In [14]:
class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1, downsample=False):
        super().__init__()

        # 1×1: 채널 축소
        self.conv1 = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=1, stride=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

        # 3×3: 공간 특징 추출
        self.conv2 = nn.Conv2d(
            out_channels, out_channels,
            kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        # 1×1: 채널 확장
        self.conv3 = nn.Conv2d(
            out_channels, self.expansion * out_channels,
            kernel_size=1, stride=1, bias=False
        )
        self.bn3 = nn.BatchNorm2d(self.expansion * out_channels)

        self.relu = nn.ReLU(inplace=True)

        if downsample:
            conv = nn.Conv2d(
                in_channels, self.expansion * out_channels,
                kernel_size=1, stride=stride, bias=False
            )
            bn = nn.BatchNorm2d(self.expansion * out_channels)
            self.downsample = nn.Sequential(conv, bn)
        else:
            self.downsample = None

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(identity)

        out += identity
        out = self.relu(out)

        return out

print(Bottleneck(64, 64, downsample=True))

Bottleneck(
  (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (downsample): Sequential(
    (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
)


In [15]:
# Bottleneck 동작 확인
x = torch.randn(2, 64, 56, 56)
b = Bottleneck(64, 64, downsample=True)
y = b(x)

print("입력:", x.shape)
print("출력:", y.shape)
print("expansion =", Bottleneck.expansion)

입력: torch.Size([2, 64, 56, 56])
출력: torch.Size([2, 256, 56, 56])
expansion = 4


### 실행 결과 해석
Bottleneck은 `1×1 → 3×3 → 1×1` 구조이며 `expansion=4`입니다.  
따라서 `out_channels=64`이면 최종 출력 채널은 `64×4=256`이 됩니다.


## 8. ResNet 전체 네트워크 정의

In [16]:
class ResNet(nn.Module):
    def __init__(self, config, output_dim, zero_init_residual=False):
        super().__init__()

        block, n_blocks, channels = config
        self.in_channels = channels[0]

        assert len(n_blocks) == len(channels) == 4

        self.conv1 = nn.Conv2d(
            3, self.in_channels,
            kernel_size=7, stride=2, padding=3, bias=False
        )
        self.bn1 = nn.BatchNorm2d(self.in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self.get_resnet_layer(
            block, n_blocks[0], channels[0]
        )
        self.layer2 = self.get_resnet_layer(
            block, n_blocks[1], channels[1], stride=2
        )
        self.layer3 = self.get_resnet_layer(
            block, n_blocks[2], channels[2], stride=2
        )
        self.layer4 = self.get_resnet_layer(
            block, n_blocks[3], channels[3], stride=2
        )

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(self.in_channels, output_dim)

        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, Bottleneck):
                    nn.init.constant_(m.bn3.weight, 0)
                elif isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)

    def get_resnet_layer(self, block, n_blocks, channels, stride=1):
        layers = []

        # stride가 1이 아니거나 채널 수가 다르면 downsample 필요
        if self.in_channels != block.expansion * channels:
            downsample = True
        else:
            downsample = False

        layers.append(
            block(self.in_channels, channels, stride, downsample)
        )

        for _ in range(1, n_blocks):
            layers.append(
                block(block.expansion * channels, channels)
            )

        self.in_channels = block.expansion * channels

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)      # 224 -> 112
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)    # 112 -> 56

        x = self.layer1(x)     # 56
        x = self.layer2(x)     # 28
        x = self.layer3(x)     # 14
        x = self.layer4(x)     # 7

        x = self.avgpool(x)    # 1×1
        x = x.view(x.shape[0], -1)
        x = self.fc(x)

        return x

print("ResNet 클래스 정의 완료")

ResNet 클래스 정의 완료


## 9. ResNet Config 정의

In [17]:
ResNetConfig = namedtuple('ResNetConfig', ['block', 'n_blocks', 'channels'])

resnet18_config = ResNetConfig(
    block=BasicBlock,
    n_blocks=[2, 2, 2, 2],
    channels=[64, 128, 256, 512]
)

resnet34_config = ResNetConfig(
    block=BasicBlock,
    n_blocks=[3, 4, 6, 3],
    channels=[64, 128, 256, 512]
)

resnet50_config = ResNetConfig(
    block=Bottleneck,
    n_blocks=[3, 4, 6, 3],
    channels=[64, 128, 256, 512]
)

resnet101_config = ResNetConfig(
    block=Bottleneck,
    n_blocks=[3, 4, 23, 3],
    channels=[64, 128, 256, 512]
)

resnet152_config = ResNetConfig(
    block=Bottleneck,
    n_blocks=[3, 8, 36, 3],
    channels=[64, 128, 256, 512]
)

print("ResNet18 blocks:", resnet18_config.n_blocks)
print("ResNet50 blocks:", resnet50_config.n_blocks)

ResNet18 blocks: [2, 2, 2, 2]
ResNet50 blocks: [3, 4, 6, 3]


## 10. 직접 구현한 ResNet50 생성

In [18]:
OUTPUT_DIM = 2   # cat / dog
model = ResNet(resnet50_config, OUTPUT_DIM)
model = model.to(device)

print(model.__class__.__name__)
print("출력 클래스 수:", model.fc.out_features)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
print(f"전체 파라미터 수: {total_params:,}")
print(f"학습 가능한 파라미터 수: {trainable_params:,}")

ResNet
출력 클래스 수: 2
전체 파라미터 수: 23,512,130
학습 가능한 파라미터 수: 23,512,130


In [19]:
# 네트워크 forward shape 확인
# batch size 1, RGB, 224×224 이미지
with torch.no_grad():
    dummy = torch.randn(1, 3, 224, 224).to(device)
    out = model(dummy)

print("dummy input :", dummy.shape)
print("model output:", out.shape)

dummy input : torch.Size([1, 3, 224, 224])
model output: torch.Size([1, 2])


### 실행 결과 해석
최종 출력 크기가 `[batch_size, 2]`이면 cat/dog 두 클래스를 분류할 수 있도록 구성된 것입니다.


## 11. torchvision의 사전 학습 ResNet50 확인

In [20]:
# 교재에서는 models.resnet50(pretrained=True)를 사용합니다.
# 최신 torchvision에서는 weights 인자를 권장하므로 호환되게 작성합니다.
try:
    pretrained_model = models.resnet50(weights=None)
except TypeError:
    pretrained_model = models.resnet50(pretrained=False)

print(pretrained_model.__class__.__name__)
print("기본 출력 클래스 수:", pretrained_model.fc.out_features)

# dogs-vs-cats에 맞게 마지막 fc를 2개 출력으로 교체
pretrained_model.fc = nn.Linear(pretrained_model.fc.in_features, 2)
pretrained_model = pretrained_model.to(device)

print("교체 후 출력 클래스 수:", pretrained_model.fc.out_features)

ResNet
기본 출력 클래스 수: 1000
교체 후 출력 클래스 수: 2


> **주의**: 실제 `pretrained=True` 또는 `weights='DEFAULT'`는 최초 실행 시 인터넷에서 가중치를 다운로드할 수 있습니다.  
> 이 제출용 노트북에서는 네트워크 구조 확인을 위해 다운로드 없이 생성하도록 했습니다.


## 12. Optimizer와 손실 함수

In [21]:
optimizer = optim.Adam(model.parameters(), lr=1e-7)
criterion = nn.CrossEntropyLoss()

model = model.to(device)
criterion = criterion.to(device)

print(optimizer)
print(criterion)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 1e-07
    maximize: False
    weight_decay: 0
)
CrossEntropyLoss()


## 13. Top-k 정확도 계산 함수

In [22]:
def calculate_topk_accuracy(y_pred, y, k=2):
    with torch.no_grad():
        batch_size = y.shape[0]

        _, top_pred = y_pred.topk(k, 1)
        top_pred = top_pred.t()

        correct = top_pred.eq(y.view(1, -1).expand_as(top_pred))

        correct_1 = correct[:1].reshape(-1).float().sum(0, keepdim=True)
        correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)

        acc_1 = correct_1 / batch_size
        acc_k = correct_k / batch_size

    return acc_1, acc_k

# 간단한 동작 확인
sample_pred = torch.tensor([[2.0, 1.0],
                            [0.1, 2.5],
                            [3.0, 0.2]])
sample_y = torch.tensor([0, 1, 1])

acc1, acc2 = calculate_topk_accuracy(sample_pred, sample_y, k=2)
print("top-1 accuracy:", acc1.item())
print("top-2 accuracy:", acc2.item())

top-1 accuracy: 0.6666666865348816
top-2 accuracy: 1.0


### 실행 결과 해석
예제에서는 3개 샘플 중 2개를 Top-1에서 올바르게 예측하여 Top-1 accuracy는 약 0.67이다. 현재 분류 문제는 cat/dog의 2-class 문제이므로 Top-2에서는 두 클래스가 모두 후보에 포함되어 accuracy가 1.0이 된다. 따라서 실제 성능을 판단할 때는 Top-1 accuracy를 중심으로 확인한다.


## 14. 모델 학습 함수

In [23]:
def train(model, iterator, optimizer, criterion, device):
    epoch_loss = 0
    epoch_acc_1 = 0
    epoch_acc_2 = 0

    model.train()

    for (x, y) in iterator:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        y_pred = model(x)
        loss = criterion(y_pred, y)

        acc_1, acc_2 = calculate_topk_accuracy(y_pred, y)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc_1 += acc_1.item()
        epoch_acc_2 += acc_2.item()

    epoch_loss /= len(iterator)
    epoch_acc_1 /= len(iterator)
    epoch_acc_2 /= len(iterator)

    return epoch_loss, epoch_acc_1, epoch_acc_2

print("train 함수 정의 완료")

train 함수 정의 완료


## 15. 모델 평가 함수

In [24]:
def evaluate(model, iterator, criterion, device):
    epoch_loss = 0
    epoch_acc_1 = 0
    epoch_acc_2 = 0

    model.eval()

    with torch.no_grad():
        for (x, y) in iterator:
            x = x.to(device)
            y = y.to(device)

            y_pred = model(x)
            loss = criterion(y_pred, y)

            acc_1, acc_2 = calculate_topk_accuracy(y_pred, y)

            epoch_loss += loss.item()
            epoch_acc_1 += acc_1.item()
            epoch_acc_2 += acc_2.item()

    epoch_loss /= len(iterator)
    epoch_acc_1 /= len(iterator)
    epoch_acc_2 /= len(iterator)

    return epoch_loss, epoch_acc_1, epoch_acc_2

print("evaluate 함수 정의 완료")

evaluate 함수 정의 완료


## 16. Epoch 시간 측정 함수

In [25]:
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

print("epoch_time 함수 정의 완료")

epoch_time 함수 정의 완료


## 17. 전체 학습 루프

In [26]:
# 실제 교재 데이터가 있을 때만 실행
if DATA_AVAILABLE:
    best_valid_loss = float('inf')
    EPOCHS = 10

    for epoch in range(EPOCHS):
        start_time = time.monotonic()

        train_loss, train_acc_1, train_acc_2 = train(
            model, train_iterator, optimizer, criterion, device
        )
        valid_loss, valid_acc_1, valid_acc_2 = evaluate(
            model, valid_iterator, criterion, device
        )

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            torch.save(model.state_dict(), './ResNet-model.pt')

        end_time = time.monotonic()
        epoch_mins, epoch_secs = epoch_time(start_time, end_time)

        print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s')
        print(f'\tTrain Loss: {train_loss:.3f} | Train Acc@1: {train_acc_1*100:6.2f}%')
        print(f'\tValid Loss: {valid_loss:.3f} | Valid Acc@1: {valid_acc_1*100:6.2f}%')
else:
    print("교재 dogs-vs-cats 데이터가 없어 실제 학습은 생략했습니다.")
    print("데이터 경로를 맞춘 뒤 이 셀을 다시 실행하면 됩니다.")

Epoch: 01 | Epoch Time: 0m 7s
	Train Loss: 0.952 | Train Acc@1:  46.63%
	Valid Loss: 0.693 | Valid Acc@1:  53.12%
Epoch: 02 | Epoch Time: 0m 6s
	Train Loss: 0.941 | Train Acc@1:  46.63%
	Valid Loss: 0.706 | Valid Acc@1:  53.12%
Epoch: 03 | Epoch Time: 0m 6s
	Train Loss: 0.919 | Train Acc@1:  47.60%
	Valid Loss: 0.729 | Valid Acc@1:  53.12%
Epoch: 04 | Epoch Time: 0m 6s
	Train Loss: 0.940 | Train Acc@1:  46.88%
	Valid Loss: 0.791 | Valid Acc@1:  53.12%
Epoch: 05 | Epoch Time: 0m 6s
	Train Loss: 0.935 | Train Acc@1:  47.12%
	Valid Loss: 0.840 | Valid Acc@1:  53.12%
Epoch: 06 | Epoch Time: 0m 6s
	Train Loss: 0.923 | Train Acc@1:  47.36%
	Valid Loss: 0.845 | Valid Acc@1:  53.12%
Epoch: 07 | Epoch Time: 0m 5s
	Train Loss: 0.918 | Train Acc@1:  47.36%
	Valid Loss: 0.843 | Valid Acc@1:  53.12%
Epoch: 08 | Epoch Time: 0m 6s
	Train Loss: 0.906 | Train Acc@1:  47.36%
	Valid Loss: 0.839 | Valid Acc@1:  53.12%
Epoch: 09 | Epoch Time: 0m 6s
	Train Loss: 0.902 | Train Acc@1:  48.08%
	Valid Loss: 0.8

### 학습 결과에 대한 주석
이번 실습에서는 전체 데이터 중 400장을 학습 데이터, 100장을 검증 데이터로 사용하여 ResNet50을 10 epoch 학습하였다.
학습 정확도는 약 45~48%, 검증 정확도는 약 53% 수준으로 나타났으며, loss 역시 큰 폭으로 감소하지 않았다.
이는 적은 학습 데이터와 매우 작은 learning rate(1e-7)를 사용하여 짧게 학습했기 때문에 모델이 충분히 학습되지 않은 결과로 볼 수 있다. 이번 실습에서는 높은 분류 정확도를 얻는 것보다 ResNet의 residual block 구조와 전체 학습 과정이 정상적으로 동작하는지 확인하는 데 초점을 두었다.


## 18. 테스트 데이터 예측 결과 저장

In [27]:
if DATA_AVAILABLE and len(test_images_filepaths) > 0:
    import pandas as pd

    id_list = []
    pred_list = []

    model.eval()

    with torch.no_grad():
        for test_path in test_images_filepaths:
            img = Image.open(test_path)
            _id = os.path.splitext(os.path.basename(test_path))[0]

            transform = ImageTransform(size, mean, std)
            img = transform(img, phase='val')
            img = img.unsqueeze(0).to(device)

            outputs = model(img)
            preds = F.softmax(outputs, dim=1)[:, 1].tolist()

            id_list.append(_id)
            pred_list.append(preds[0])

    res = pd.DataFrame({'id': id_list, 'label': pred_list})
    print(res.head(10))

    res.to_csv('./ResNet.csv', index=False)
    print("ResNet.csv 저장 완료")
else:
    print("테스트 이미지가 없어 예측 저장 셀은 생략했습니다.")

      id     label
0   8407  0.256568
1   7533  0.263381
2  12371  0.152740
3   2576  0.325615
4   7624  0.282100
5   6418  0.209975
6    624  0.250858
7   3004  0.221991
8   6301  0.250573
9   7751  0.322591
ResNet.csv 저장 완료


## 19. 실습 정리

이번 실습에서 확인한 내용은 다음과 같습니다.

1. **Residual connection**은 합성곱 출력 `F(x)`에 입력 `x`를 더하는 구조입니다.
2. 입력과 출력 차원이 다르면 **1×1 convolution**으로 shortcut의 차원을 맞춥니다.
3. `BasicBlock`은 ResNet18/34에서 사용되고, `Bottleneck`은 ResNet50/101/152에서 사용됩니다.
4. Bottleneck은 `1×1 → 3×3 → 1×1` 구조로 계산량을 줄이면서 깊은 네트워크를 구성합니다.
5. `ResNetConfig`를 통해 block 종류와 각 stage의 block 수를 바꾸면 여러 ResNet을 만들 수 있습니다.
6. 직접 구현한 ResNet50의 출력 shape가 `[batch_size, 2]`가 되는 것을 확인했습니다.

### 배운 점
ResNet의 핵심은 단순히 층을 많이 쌓는 것이 아니라, shortcut을 통해 입력 정보를 직접 전달하면서 residual만 학습하도록 구조를 바꾸는 것이라는 점을 코드로 확인할 수 있었습니다.실제 학습에서는 데이터 수와 learning rate 등의 학습 조건이 성능에 큰 영향을 주기 때문에, residual 구조가 구현되었다는 사실과 높은 분류 정확도를 얻는 것은 별개의 문제임을 확인하였습니다.
